# Imports

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(188)

# Download dataset

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-07 04:40:41--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-05-07 04:40:41 (39.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



# Previous Implementation

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 100
eval_interval = 10
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def precompute_rope_params(n_embd, theta_base=10_000, block_size=4096):
    assert n_embd % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, n_embd, 2)[: (n_embd // 2)].float() / n_embd))

    # Generate position indices
    positions = torch.arange(block_size)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (block_size, n_embd // 2)

    # Expand angles to match the n_embd
    angles = torch.cat([angles, angles], dim=1)  # Shape: (block_size, n_embd)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def compute_rope(x, cos, sin, start_pos=0):
    # x: (batch_size, num_heads, seq_len, n_embd)
    batch_size, seq_len, n_embd = x.shape
    assert n_embd % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : n_embd // 2]  # First half
    x2 = x[..., n_embd // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[start_pos : start_pos+seq_len, :].unsqueeze(0)  # Shape: (1, seq_len, head_dim)
    sin = sin[start_pos : start_pos+seq_len, :].unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    return  (x * cos) + (rotated * sin)


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        ################################### ROPE PARAMS ############################
        cos, sin = precompute_rope_params(n_embd=n_embd//n_head, block_size=block_size)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        ###########################################################################

        ################################### KV CACHE ###############################
        self.register_buffer("cache_k", None)
        self.register_buffer("cache_v", None)
        ###########################################################################

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k_new = self.key(x)   # (B,T,hs)
        v_new = self.value(x) # (B,T,hs)
        q = self.query(x) # (B,T,hs)

        #Apply RoPE embeddings
        ################################### NEW ###################################
        k_new = compute_rope(k_new, self.cos, self.sin, start_pos=start_pos)
        q = compute_rope(q, self.cos, self.sin, start_pos=start_pos)
        ###########################################################################

        if use_cache:
            if self.cache_k is None:
                self.cache_k = k_new
                self.cache_v = v_new
            else:
                self.cache_k = torch.cat([self.cache_k, k_new], dim=1)
                self.cache_v = torch.cat([self.cache_v, v_new], dim=1)
            k, v = self.cache_k, self.cache_v
        else:
            k, v = k_new, v_new


        # compute attention scores ("affinities")
        T_q, T_k = q.shape[1], k.shape[1]
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[T_k-T_q:T_k, :T_k] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        return wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        out = torch.cat([h(x, use_cache=use_cache, start_pos=start_pos) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.SiLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

    def forward(self, x, use_cache=False, start_pos=0):
        x = x + self.sa(self.ln1(x), use_cache=use_cache, start_pos=start_pos)
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape

        if use_cache:
            # Determine how many tokens are already cached
            first_head = self.blocks[0].sa.heads[0]
            cached_len = 0 if first_head.cache_k is None else first_head.cache_k.shape[1]
            # Only process tokens not yet in cache
            idx = idx[:, cached_len:]
            start_pos = cached_len
        else:
            start_pos = 0

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        for block in self.blocks:
            x = block(x, use_cache=use_cache, start_pos=start_pos)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def clear_cache(self):
        for block in self.blocks:
            for head in block.sa.heads:
                head.cache_k = None
                head.cache_v = None

    def generate(self, idx, max_new_tokens, use_cache=True):
        if use_cache:
            self.clear_cache()
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop to block_size to stay within the causal mask bounds.
            # NOTE: when use_cache=True and total sequence length (initial context +
            # generated tokens) exceeds block_size, this crop and the cached_len tracked
            # in forward() get out of sync — cached_len is an absolute position counter
            # but idx_cond is a relative sliding window. This causes wrong start_pos values
            # and therefore incorrect RoPE encodings for new tokens. Safe for short
            # generations; for sequences longer than block_size a sliding-window cache
            # with eviction of old entries would be needed.
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, _ = self(idx_cond, use_cache=use_cache)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# Change MHA to use Single Matrix to maintain mulitple heads

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 100
eval_interval = 10
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def precompute_rope_params(head_dim, theta_base=10_000, block_size=4096):
    assert head_dim % 2 == 0, "Head dimension must be even"
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
    positions = torch.arange(block_size)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    angles = torch.cat([angles, angles], dim=1)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

def compute_rope(x, cos, sin, start_pos=0):
    # x: (B, n_heads, T, head_dim)
    seq_len = x.shape[2]
    head_dim = x.shape[3]
    assert head_dim % 2 == 0, "Head dimension must be even"

    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]

    cos = cos[start_pos:start_pos+seq_len, :].view(1, 1, seq_len, head_dim)
    sin = sin[start_pos:start_pos+seq_len, :].view(1, 1, seq_len, head_dim)

    rotated = torch.cat((-x2, x1), dim=-1)
    return (x * cos) + (rotated * sin)


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_embd = n_embd
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope_params(head_dim=self.head_size, block_size=block_size)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        B, T, C = x.shape

        # Project Q, K, V
        q = self.q_proj(x)  # (B, T, n_embd)
        k = self.k_proj(x)  # (B, T, n_embd)
        v = self.v_proj(x)  # (B, T, n_embd)

        # Reshape to multi-head format
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, n_head, T, hs)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, n_head, T, hs)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, n_head, T, hs)

        # Apply RoPE to Q and K
        q = compute_rope(q, self.cos, self.sin, start_pos=start_pos)
        k = compute_rope(k, self.cos, self.sin, start_pos=start_pos)

        # KV Cache
        if use_cache:
            if self.cache_k is None:
                self.cache_k = k
                self.cache_v = v
            else:
                self.cache_k = torch.cat([self.cache_k, k], dim=2)
                self.cache_v = torch.cat([self.cache_v, v], dim=2)
            k, v = self.cache_k, self.cache_v

        # Compute attention scores
        T_q, T_k = q.shape[2], k.shape[2]
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5  # (B, n_head, T_q, T_k)
        wei = wei.masked_fill(
            self.tril[T_k-T_q:T_k, :T_k].unsqueeze(0).unsqueeze(0) == 0,
            float('-inf')
        )
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Weighted aggregation
        out = wei @ v  # (B, n_head, T_q, hs)
        out = out.transpose(1, 2).contiguous().view(B, T_q, self.n_head * self.head_size)
        return self.out_proj(out)


class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.SiLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

    def forward(self, x, use_cache=False, start_pos=0):
        x = x + self.sa(self.ln1(x), use_cache=use_cache, start_pos=start_pos)
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape

        if use_cache:
            # Determine how many tokens are already cached
            cached_len = 0 if self.blocks[0].sa.cache_k is None else self.blocks[0].sa.cache_k.shape[2]
            # Only process tokens not yet in cache
            idx = idx[:, cached_len:]
            start_pos = cached_len
        else:
            start_pos = 0

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        for block in self.blocks:
            x = block(x, use_cache=use_cache, start_pos=start_pos)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def clear_cache(self):
        for block in self.blocks:
            block.sa.cache_k = None
            block.sa.cache_v = None

    def generate(self, idx, max_new_tokens, use_cache=True):
        if use_cache:
            self.clear_cache()
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop to block_size to stay within the causal mask bounds
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, _ = self(idx_cond, use_cache=use_cache)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.683329 M parameters


## Training

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## Inference

In [ ]:
import time

context = torch.zeros((1, 1), dtype=torch.long, device=device)
NUM_TOKENS = 200

model.eval()

# Time WITHOUT cache
with torch.no_grad():
    t0 = time.time()
    out_no_cache = model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=False)
    t1 = time.time()
    time_no_cache = t1 - t0

# Time WITH cache
with torch.no_grad():
    t0 = time.time()
    out_cache = model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=True)
    t1 = time.time()
    time_cache = t1 - t0

print(f"Without cache: {time_no_cache:.2f}s  ({NUM_TOKENS/time_no_cache:.1f} tok/s)")
print(f"With cache:    {time_cache:.2f}s  ({NUM_TOKENS/time_cache:.1f} tok/s)")
print(f"Speedup: {time_no_cache/time_cache:.2f}x")
print()
print("--- Sample output (with cache) ---")
print(decode(out_cache[0].tolist()))

Without cache: 7.30s  (27.4 tok/s)
With cache:    3.79s  (52.8 tok/s)
Speedup: 1.93x

--- Sample output (with cache) ---

Hcg&cQ vgWCw'VNT'iJDN'hw.nYjMjcfc
Rd-MZGiljk -sXj&G 3xu-pPG;XpEsO;:-fBQTigXXZ?bR-ogLMr;&,WWsdPUgBxSIVh&lp3UkEIIfDzSMASXLZpzhxa;&-ihDMc$!NIwVGkuvIw &qxmBWIROP,.gOjHbQ?EEb3PegeToJJOl',gQRIOraaKoNZw $l;d


# Llama Model with Grouped-Query Attention

## Understanding Grouped-Query Attention (GQA)

In the previous notebooks we used **Multi-Head Attention (MHA)** where every query head has its own dedicated Key and Value projections. This works well but becomes a memory bottleneck at scale — the KV cache grows linearly with the number of heads.

**Grouped-Query Attention (GQA)** is the solution used by Llama 2 70B, Llama 3, Mistral, and Gemma. It sits between two extremes:

| Variant | Q heads | KV heads | KV cache size |
|---------|---------|----------|---------------|
| **MHA** (Multi-Head Attention) | `n_head` | `n_head` | Full |
| **GQA** (Grouped-Query Attention) | `n_head` | `n_kv_head` | Reduced by `n_head / n_kv_head` |
| **MQA** (Multi-Query Attention) | `n_head` | 1 | Minimal |

### How it works

With `n_head=6` query heads and `n_kv_head=2` KV heads, every **3 query heads share 1 KV head**:

```
Query heads:   [Q0] [Q1] [Q2]   [Q3] [Q4] [Q5]
                 \   |   /         \   |   /
KV heads:        [K0, V0]          [K1, V1]
```

### Why it matters

- **KV cache memory** drops by `n_head / n_kv_head` (3x in our case)
- **Fewer parameters** in K/V projections
- **Nearly matches MHA quality** — unlike MQA which can degrade
- At scale (billions of params), this is the difference between fitting in GPU memory or not

### Implementation change

In our previous notebooks, attention used separate `Head` modules in a `ModuleList`. For GQA, we replace this with a single `GroupedQueryAttention` module that:
1. Projects Q with `n_head` output heads, but K and V with only `n_kv_head` output heads
2. Uses `repeat_interleave` to expand KV heads to match Q heads before computing attention
3. Stores only the compact `n_kv_head` tensors in the KV cache

In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 100
eval_interval = 10
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_kv_head = 2     # number of key/value heads (GQA: shared across groups of query heads)
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def precompute_rope_params(head_dim, theta_base=10_000, block_size=4096):
    assert head_dim % 2 == 0, "Head dimension must be even"
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
    positions = torch.arange(block_size)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    angles = torch.cat([angles, angles], dim=1)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

def compute_rope(x, cos, sin, start_pos=0):
    # x: (B, n_heads, T, head_dim)
    seq_len = x.shape[2]
    head_dim = x.shape[3]
    assert head_dim % 2 == 0, "Head dimension must be even"

    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]

    cos = cos[start_pos:start_pos+seq_len, :].view(1, 1, seq_len, head_dim)
    sin = sin[start_pos:start_pos+seq_len, :].view(1, 1, seq_len, head_dim)

    rotated = torch.cat((-x2, x1), dim=-1)
    return (x * cos) + (rotated * sin)


class GQAAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_embd = n_embd
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.n_kv_head = n_kv_head
        self.n_group = self.n_head // self.n_kv_head
        self.q_proj = nn.Linear(n_embd, self.head_size * self.n_head, bias=False)
        self.k_proj = nn.Linear(n_embd, self.head_size * self.n_kv_head, bias=False)
        self.v_proj = nn.Linear(n_embd, self.head_size * self.n_kv_head, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        cos, sin = precompute_rope_params(head_dim=self.head_size, block_size=block_size)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        B, T, C = x.shape

        # Project Q, K, V
        q = self.q_proj(x)  # (B, T, n_embd)
        k = self.k_proj(x)  # (B, T, n_embd)
        v = self.v_proj(x)  # (B, T, n_embd)

        # Reshape to multi-head format
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, n_head, T, hs)
        k = k.view(B, T, self.n_kv_head, self.head_size).transpose(1, 2)  # (B, n_kv_head, T, hs)
        v = v.view(B, T, self.n_kv_head, self.head_size).transpose(1, 2)  # (B, n_kv_head, T, hs)

        # Apply RoPE to Q and K
        q = compute_rope(q, self.cos, self.sin, start_pos=start_pos)
        k = compute_rope(k, self.cos, self.sin, start_pos=start_pos)

        # KV Cache
        if use_cache:
            if self.cache_k is None:
                self.cache_k = k
                self.cache_v = v
            else:
                self.cache_k = torch.cat([self.cache_k, k], dim=2)
                self.cache_v = torch.cat([self.cache_v, v], dim=2)
            k, v = self.cache_k, self.cache_v

        
        ################################### GQA KEY STEP ###########################
        # Expand KV heads to match query heads by repeating each KV head n_group times
        # (B, n_kv_head, T_k, hs) -> (B, n_head, T_k, hs)
        k = k.repeat_interleave(self.n_group, dim=1)
        v = v.repeat_interleave(self.n_group, dim=1)
        ###########################################################################

        # Compute attention scores
        T_q, T_k = q.shape[2], k.shape[2]
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5  # (B, n_head, T_q, T_k)
        wei = wei.masked_fill(
            self.tril[T_k-T_q:T_k, :T_k].unsqueeze(0).unsqueeze(0) == 0,
            float('-inf')
        )
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Weighted aggregation
        out = wei @ v  # (B, n_head, T_q, hs)
        out = out.transpose(1, 2).contiguous().view(B, T_q, self.n_head * self.head_size)
        return self.out_proj(out)


class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.SiLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        self.sa = GQAAttention(n_embd, n_head)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

    def forward(self, x, use_cache=False, start_pos=0):
        x = x + self.sa(self.ln1(x), use_cache=use_cache, start_pos=start_pos)
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel_GQA(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape

        if use_cache:
            # Determine how many tokens are already cached
            cached_len = 0 if self.blocks[0].sa.cache_k is None else self.blocks[0].sa.cache_k.shape[2]
            # Only process tokens not yet in cache
            idx = idx[:, cached_len:]
            start_pos = cached_len
        else:
            start_pos = 0

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        for block in self.blocks:
            x = block(x, use_cache=use_cache, start_pos=start_pos)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def clear_cache(self):
        for block in self.blocks:
            block.sa.cache_k = None
            block.sa.cache_v = None

    def generate(self, idx, max_new_tokens, use_cache=True):
        if use_cache:
            self.clear_cache()
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop to block_size to stay within the causal mask bounds
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, _ = self(idx_cond, use_cache=use_cache)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [15]:
m = GPTLanguageModel_GQA()

In [16]:
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

9.503681 M parameters


## Parameter Comparison: MHA vs GQA

In [ ]:
head_size = n_embd // n_head

# MHA: each of n_head heads has its own Q, K, V projection
mha_kv_params = 2 * n_head * (n_embd * head_size)
# GQA: only n_kv_head K and V projections
gqa_kv_params = 2 * n_kv_head * (n_embd * head_size)

print(f"Config: n_head={n_head}, n_kv_head={n_kv_head}, head_size={head_size}")
print(f"  Query heads per KV group: {n_head // n_kv_head}")
print()
print(f"MHA K/V params per layer:  {mha_kv_params:>10,}")
print(f"GQA K/V params per layer:  {gqa_kv_params:>10,}")
print(f"K/V param reduction:       {(1 - gqa_kv_params/mha_kv_params)*100:.1f}%")
print()
print(f"MHA KV cache per token per layer: {2 * n_head * head_size} floats")
print(f"GQA KV cache per token per layer: {2 * n_kv_head * head_size} floats")
print(f"KV cache reduction:              {(1 - n_kv_head/n_head)*100:.1f}%")
print()

mha_total = sum(p.numel() for p in m.parameters()) + (mha_kv_params - gqa_kv_params) * n_layer
gqa_total = sum(p.numel() for p in m.parameters())
print(f"Estimated total model params with MHA: {mha_total/1e6:.2f}M")
print(f"Actual total model params with GQA:    {gqa_total/1e6:.2f}M")
print(f"Overall param reduction:               {(1 - gqa_total/mha_total)*100:.1f}%")

## Training

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## Inference

In [17]:
import time

context = torch.zeros((1, 1), dtype=torch.long, device=device)
NUM_TOKENS = 200

GQA_model = GPTLanguageModel_GQA()
MHA_model = GPTLanguageModel()

GQA_model.eval()
MHA_model.eval()

# Time with MHA
with torch.no_grad():
    t0 = time.time()
    out_mha = MHA_model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=False)
    t1 = time.time()
    time_mha = t1 - t0

# Time with GQA
with torch.no_grad():
    t0 = time.time()
    out_gqa = GQA_model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=True)
    t1 = time.time()
    time_gqa = t1 - t0

print(f"With MHA: {time_mha:.2f}s  ({NUM_TOKENS/time_mha:.1f} tok/s)")
print(f"With GQA:    {time_gqa:.2f}s  ({NUM_TOKENS/time_gqa:.1f} tok/s)")
print(f"Speedup: {time_mha/time_gqa:.2f}x")

# KV Cache size comparison
sa = GQA_model.blocks[0].sa
seq_len     = sa.cache_k.shape[2]
elem_bytes  = sa.cache_k.element_size()

gqa_per_layer = (sa.cache_k.numel() + sa.cache_v.numel()) * elem_bytes
gqa_total     = gqa_per_layer * n_layer

mha_per_layer = gqa_per_layer * (n_head / n_kv_head)
mha_total     = mha_per_layer * n_layer

print()
print(f"--- KV Cache Size ({seq_len} tokens, {n_layer} layers) ---")
print(f"MHA: {int(mha_total):>8,} bytes  ({mha_total/1024:.2f} KB)")
print(f"GQA: {gqa_total:>8,} bytes  ({gqa_total/1024:.2f} KB)")
print(f"Cache reduction: {(1 - gqa_total/mha_total)*100:.1f}%  ({n_head//n_kv_head}x smaller)")

With MHA: 0.99s  (201.7 tok/s)
With GQA:    0.25s  (799.4 tok/s)
Speedup: 3.96x

--- KV Cache Size (200 tokens, 6 layers) ---
MHA: 3,686,400 bytes  (3600.00 KB)
GQA: 1,228,800 bytes  (1200.00 KB)
Cache reduction: 66.7%  (3x smaller)
